# 09 — Собственные задачи и верификация (v0.6.4)

Девятый ноутбук цепочки: он не вводит новую геометрию, а показывает **два новых
типа анализа** и **как комплекс проверяет сам себя**.

* **Собственные задачи** — устойчивость (потеря устойчивости) и свободные
  колебания; плюс **преднапряжённые** колебания *нагруженной* пластины.
* **Ньютон-ускорение** нелинейной итерации (Карман/КТН).
* **Лестница верификации** полной нелинейной КТН без литературных эталонов:
  редукции R1–R5, метод изготовленных решений (MMS) и независимый радиальный
  решатель (1D↔2D, ворота R3).

Все числа воспроизводимы: `E = a = 1`, безразмерная нагрузка `P̄ = q₀/h⁴`.

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from plate_solver.config import Config
from plate_solver.eigenmodes import buckling, linear_plate, natural_frequencies
from plate_solver.geometry import make_circle, make_rectangle
from plate_solver.ktn_full import KTNPlate
from plate_solver.membrane import KarmanPlate

matplotlib.use("Agg")                      # безголовое исполнение
np.set_printoptions(precision=4, suppress=True)

## 1. Устойчивость и колебания

Оба анализа — обобщённые собственные задачи поверх ТОЙ ЖЕ линейной сборки
(изгибная жёсткость `K = D·S_bend`, геометрическая `K_geo(N)`, масса `M = ρh·∫ψψ`).

* **Потеря устойчивости:** `(K + λ·K_geo(N⁰))φ = 0` — критический множитель `λ_cr`.
* **Колебания:** `Kφ = ω²Mφ`.

Проверяем по классическим эталонам Кирхгофа (они есть в литературе — Тимошенко, Лейсса).

In [2]:
A = 1.0
cfg = Config(E=1.0, nu=0.3, h=0.01, q0=0.0, p=12, Q=72)
D = cfg.D

# Устойчивость: квадрат SSSS, одноосное сжатие -> k = N_cr b^2 / (pi^2 D) = 4 (Тимошенко)
sq = linear_plate(make_rectangle(0, A, 0, A), cfg, bc_type="soft_hinge")
lam = buckling(sq, Nx=-1.0, n_modes=1).values[0]
k = lam * A**2 / (np.pi**2 * D)
print(f"Устойчивость SSSS-квадрата:  k = {k:.4f}   (эталон Тимошенко: 4.000)")

# Колебания: круг CCCC, фундаментальный частотный параметр (Лейсса 10.2158)
disk = linear_plate(make_circle(A), cfg, bc_type="clamped")
w = natural_frequencies(disk, n_modes=1).values[0]
lam1 = w * A**2 * np.sqrt(1.0 / D)
print(f"Колебания круга CCCC:        λ₁ = {lam1:.4f}   (эталон Лейсса: 10.216)")

Устойчивость SSSS-квадрата:  k = 4.0002   (эталон Тимошенко: 4.000)
Колебания круга CCCC:        λ₁ = 10.2591   (эталон Лейсса: 10.216)


In [3]:
# Формы: 1-я форма выпучивания квадрата и 1-я форма колебаний круга
buck = buckling(sq, Nx=-1.0, n_modes=1)
vib = natural_frequencies(disk, n_modes=1)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
panels = ((ax[0], buck, "форма выпучивания (SSSS-квадрат)"),
          (ax[1], vib, "форма колебаний (круг CCCC)"))
for a_, res, title in panels:
    Xg, Yg, Wm = res.mode_on_grid(0, grid_n=80)
    a_.contourf(Xg, Yg, Wm, levels=20, cmap="RdBu_r")
    a_.set_title(title)
    a_.set_aspect("equal")
    a_.axis("off")
plt.tight_layout()
plt.show()

/var/folders/7d/l76g4yv93kd285y42qwzx0pm0000gp/T/ipykernel_50767/2306518594.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


То же — прямо из **case-файла** (секция `[eigen]`), через штатный диспетчер:

In [4]:
from plate_solver import dispatch
from plate_solver.problem import Problem

for case in ("cases/ci/eigen_vibration_square.toml",
             "cases/ci/eigen_buckling_circle.toml"):
    r = dispatch.solve(Problem.from_toml(case))
    print(f"{case.split('/')[-1]:30s}: {r.eigen.kind:9s}  первые значения = {r.eigen.values[:3]}")

eigen_vibration_square.toml   : vibration  первые значения = [ 8.6566 21.6413 21.6413]
eigen_buckling_circle.toml    : buckling   первые значения = [2.8375 5.0983 5.0983]


## 2. Преднапряжённые колебания — частоты *нагруженной* пластины

Под поперечной нагрузкой срединная поверхность **растягивается**; кармановское
натяжение `N(w) > 0` ужесточает пластину. Подставляя РЕАЛЬНОЕ поле `N(w)` из
нелинейного решения как преднапряжение (`K_eff = K + K_geo(N(w))`), получаем
**рост фундаментальной частоты с нагрузкой** — физический эффект, а не артефакт.

In [5]:
dom = make_circle(1.0)
base = dict(E=1.0, nu=0.3, h=0.1, a=1.0, p=12, Q=140,
            n_load_steps=3, karman_tol=1e-9, karman_max_iter=400)
w0 = natural_frequencies(linear_plate(dom, Config(q0=0.0, **base), bc_type="clamped"),
                         n_modes=1).values[0]
Pbars = [0.0, 2.0, 4.0, 8.0, 12.0]
freqs = [w0]
for P in Pbars[1:]:
    kp = KarmanPlate.from_config(dom, Config(q0=P*0.1**4, **base),
                                 bc_type="clamped", inplane_bc="immovable")
    res = kp.solve_uniform()
    freqs.append(natural_frequencies(kp, n_modes=1, prestress=res).values[0])
freqs = np.array(freqs)
for P, f in zip(Pbars, freqs, strict=True):
    print(f"P̄ = {P:4.1f}:  ω₁ = {f:.5f}   ({f/w0:.3f}× ненапряжённой)")

plt.figure(figsize=(5, 3.4))
plt.plot(Pbars, freqs / w0, "o-")
plt.xlabel("безразмерная нагрузка  P̄")
plt.ylabel("ω₁ / ω₁(0)")
plt.title("Натяжение от нагрузки повышает частоту")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

P̄ =  0.0:  ω₁ = 0.09802   (1.000× ненапряжённой)
P̄ =  2.0:  ω₁ = 0.10058   (1.026× ненапряжённой)
P̄ =  4.0:  ω₁ = 0.10602   (1.082× ненапряжённой)
P̄ =  8.0:  ω₁ = 0.11759   (1.200× ненапряжённой)
P̄ = 12.0:  ω₁ = 0.12773   (1.303× ненапряжённой)


/var/folders/7d/l76g4yv93kd285y42qwzx0pm0000gp/T/ipykernel_50767/4141213120.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Ньютон-ускорение нелинейной итерации

Согласованный касательный оператор `J = dR/dc` (с защитой линейным поиском) даёт
**квадратичную** сходимость — единицы итераций против десятков/сотен у Пикара, при
том же решении. Включается ключом `karman_method`/`ktn_method = "newton"`.

In [6]:
def solve_both(theory, Pbar=8.0, h=1.0):
    key = "karman_method" if theory == "karman" else "ktn_method"
    Cls = KarmanPlate if theory == "karman" else KTNPlate
    out = {}
    for method in ("picard", "newton"):
        cfg = Config(E=1.0, nu=0.3, h=h, a=1.0, q0=Pbar*h**4, p=10, Q=128,
                     karman_tol=1e-9, karman_max_iter=400, **{key: method})
        r = Cls.from_config(make_circle(1.0), cfg, bc_type="clamped",
                            inplane_bc="immovable").solve_uniform()
        out[method] = (r.n_iter, r.w_max)
    return out

for theory in ("karman", "ktn_full"):
    o = solve_both(theory)
    print(f"{theory:9s} (P̄=8): Пикар {o['picard'][0]:3d} ит | "
          f"Ньютон {o['newton'][0]:2d} ит | "
          f"|Δw|/w = {abs(o['newton'][1]-o['picard'][1])/o['picard'][1]:.1e}")

karman    (P̄=8): Пикар  79 ит | Ньютон  6 ит | |Δw|/w = 1.1e-08


ktn_full  (P̄=8): Пикар 102 ит | Ньютон  5 ит | |Δw|/w = 5.5e-09


## 4. Лестница верификации полной КТН (без литературных эталонов)

У полной нелинейной КТН нет замкнутых эталонов, поэтому корректность метода
удостоверяется **вырождением** и **независимой перепроверкой**:

* **R1–R5** (`tests/test_ktn_full.py`) — редукции: КТН→Карман (машинно), →Кирхгоф
  (тонкая пластина), гашение поправки `O(h²)`, смыкание лицевых с `ktn_linear`.
* **MMS** (`ladder.mms_ktn_load_and_exact`) — изготовленное решение при
  замороженном `N` проверяет СБОРКУ оператора до машинной точности.
* **R3, 1D↔2D** (`radial_ktn.RadialKTN`) — НЕЗАВИСИМЫЙ радиальный решатель
  (иная дискретизация) воспроизводит 2D-решение с ЖИВОЙ связью `N(w)`.

Ниже — R3 вживую: 2D-решение полной КТН на круге и независимая радиальная кривая.

In [7]:
from plate_solver.radial_ktn import RadialKTN

h, Pbar = 0.1, 6.0
rad = RadialKTN(1.0, 1.0, 0.3, h, include_ktn=True, bc="clamped")
r1 = rad.solve(Pbar*h**4)
plate = KTNPlate.from_config(make_circle(1.0),
    Config(E=1.0, nu=0.3, h=h, a=1.0, q0=Pbar*h**4, p=14, Q=160,
           n_load_steps=3, karman_tol=1e-10, karman_max_iter=400, ktn_method="newton"),
    bc_type="clamped", inplane_bc="immovable")
r2 = plate.solve_uniform()

rs = np.linspace(0, 1, 60)
w_1d = rad.deflection(r1.cw, rs)
w_2d = np.array([float(plate.deflection(r2.cw, x, 0.0)) for x in rs])
rel = abs(r1.w_max - r2.w_max) / r2.w_max
print(f"R3 (полная КТН, круг, защемление):  1D w_max = {r1.w_max:.6e}")
print(f"                                    2D w_max = {r2.w_max:.6e}")
print(f"      согласие 1D↔2D = {rel:.2e}  (дискретизационно-ограничено, убывает с 2D-p)")

plt.figure(figsize=(5.2, 3.4))
plt.plot(rs, w_2d, "o", ms=3, label="2D KTNPlate (RFM)")
plt.plot(rs, w_1d, "-", label="1D радиальный (независимый)")
plt.xlabel("r")
plt.ylabel("w(r)")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Ворота R3: две независимые дискретизации полной КТН")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

R3 (полная КТН, круг, защемление):  1D w_max = 7.658370e-02
                                    2D w_max = 7.632984e-02
      согласие 1D↔2D = 3.33e-03  (дискретизационно-ограничено, убывает с 2D-p)


/var/folders/7d/l76g4yv93kd285y42qwzx0pm0000gp/T/ipykernel_50767/4009877987.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Итог.** Комплекс не только решает четыре теории (изгиб и контакт) на областях
произвольного очертания, но и **проверяет себя**: классические эталоны там, где они
есть (устойчивость, колебания, Кирхгоф, кармановские кривые), и редукции + MMS +
независимый 1D↔2D-решатель там, где эталонов нет (полная КТН). Числа воспроизводимы
`scripts/reproduce_all.py`; ворота держит `pytest`.